# Core species analysis - GTDB only

In [1]:
import polars as pl

In [2]:
df = (
    pl.scan_parquet('../outputs.cds3/pq/gtdb.cds3.x.3216.manysearch.parquet')
    .filter(pl.col('intersect_hashes') >= 20)
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).collect()

In [3]:
df

query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes,species_name
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str
"""GCA_963606895 s__Methanocatell…","""e1e78e249a6a2a11a5d4838d46832e…","""SRR11183344""",0.224552,514,21,1000,"""DNA""","""1aa505ca6057e836a94c056022f4e0…",0.002378,0.224552,29.357977,33.0,17.309392,0.931344,0.750286,0.840815,0.931344,15090,995049,"""s__Methanocatella smithii"""
"""GCF_902163005 s__Enterococcus …","""237581605496a55a4685225090feb3…","""SRR11183344""",0.027679,443,21,1000,"""DNA""","""1aa505ca6057e836a94c056022f4e0…",0.001926,0.027679,5.997743,6.0,3.259644,0.842979,0.744993,0.793986,0.842979,2657,995049,"""s__Enterococcus faecalis"""
"""GCF_001256715 s__Escherichia c…","""1f70baaf479ba33ceb65afba904327…","""SRR11183344""",0.005168,606,21,1000,"""DNA""","""1aa505ca6057e836a94c056022f4e0…",0.001831,0.005168,6.194719,5.0,5.865802,0.778231,0.756192,0.767211,0.778231,3754,995049,"""s__Escherichia coli"""
"""GCA_034118145 s__Butyricimonas…","""d41465962f39cbe33f972d600887b7…","""SRR11183344""",0.181149,492,21,1000,"""DNA""","""1aa505ca6057e836a94c056022f4e0…",0.002271,0.181149,3.254065,3.0,2.11808,0.921867,0.748724,0.835296,0.921867,1601,995049,"""s__Butyricimonas virosa"""
"""GCA_905212675 s__Methanomethyl…","""202150856dd3473c678da6a3c48206…","""SRR11183344""",0.073897,124,21,1000,"""DNA""","""1aa505ca6057e836a94c056022f4e0…",0.000574,0.073897,1.443548,1.0,0.826013,0.883335,0.701165,0.79225,0.883335,179,995049,"""s__Methanomethylophilus alvi"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GCA_021618115 s__HGM11372 sp90…","""56d4fe6e5ac7a73f0baf4fca85b5ee…","""SRR17241661""",0.257359,306,21,1000,"""DNA""","""9291c9452ee4ff8353ec7fc637f7d3…",0.000124,0.257359,2.202614,2.0,1.390253,0.937412,0.651585,0.794498,0.937412,674,14000869,"""s__HGM11372 sp900767715"""
"""GCA_022769785 s__Anaerobutyric…","""98c316eb101ac31a2e7ed7b7ecb712…","""SRR17241661""",0.317528,375,21,1000,"""DNA""","""9291c9452ee4ff8353ec7fc637f7d3…",0.000152,0.317528,1.805333,1.0,1.667565,0.946837,0.657925,0.802381,0.946837,677,14000869,"""s__Anaerobutyricum sp022769785"""
"""GCA_902462135 s__Jutongia sp93…","""ba91244e61d2de47e91a38676bd14f…","""SRR17241661""",0.029575,32,21,1000,"""DNA""","""9291c9452ee4ff8353ec7fc637f7d3…",0.000013,0.029575,1.625,1.0,0.992157,0.845642,0.585163,0.715403,0.845642,52,14000869,"""s__Jutongia sp934339135"""


In [4]:
n_acc = df['match_name'].n_unique()
by_species = df.group_by('species_name').agg(
    freq=pl.len() / n_acc
)

In [5]:
SPECIES_ABOVE_95 = set(by_species.filter(pl.col('freq') >= 0.95)['species_name'])

In [6]:
CORE_NAMES=set([ x.strip() for x in open('../inputs.branchwater/names.list') ])

In [7]:
CORE_NAMES

{'s__Bariatricus sp004560705',
 's__Colivicinus sp002299675',
 's__Cryptobacteroides sp000432655',
 's__Cryptobacteroides sp000434935',
 's__Cryptobacteroides sp034089285',
 's__Cryptobacteroides sp900546925',
 's__Fimisoma sp002320005',
 's__Floccifex porci',
 's__JAFBIX01 sp021531895',
 's__Lactobacillus amylovorus',
 's__Mogibacterium_A kristiansenii',
 's__Ornithospirochaeta sp022785155',
 's__Prevotella sp000434975',
 's__Prevotella sp002251295',
 's__Sodaliphilus sp004557565',
 's__UBA2868 sp004552595'}

In [8]:
len(CORE_NAMES.intersection(SPECIES_ABOVE_95))

11

In [9]:
print(CORE_NAMES - SPECIES_ABOVE_95)
len(CORE_NAMES - SPECIES_ABOVE_95)

{'s__Floccifex porci', 's__Bariatricus sp004560705', 's__JAFBIX01 sp021531895', 's__UBA2868 sp004552595', 's__Mogibacterium_A kristiansenii'}


5

In [10]:
print(SPECIES_ABOVE_95 - CORE_NAMES)
len(SPECIES_ABOVE_95 - CORE_NAMES)

{'s__Escherichia coli', 's__Prevotella sp900548195', 's__CAG-170 sp002404795'}


3

In [11]:
with pl.Config(tbl_rows=-1):
    print(by_species.sort('freq', descending=True).filter(pl.col('freq') >= 0.95))

shape: (14, 2)
┌─────────────────────────────────┬──────────┐
│ species_name                    ┆ freq     │
│ ---                             ┆ ---      │
│ str                             ┆ f64      │
╞═════════════════════════════════╪══════════╡
│ s__Escherichia coli             ┆ 0.994403 │
│ s__Sodaliphilus sp004557565     ┆ 0.990361 │
│ s__Cryptobacteroides sp9005469… ┆ 0.978856 │
│ s__Cryptobacteroides sp0340892… ┆ 0.978856 │
│ s__Lactobacillus amylovorus     ┆ 0.976368 │
│ s__Fimisoma sp002320005         ┆ 0.973881 │
│ s__Cryptobacteroides sp0004349… ┆ 0.968595 │
│ s__Prevotella sp002251295       ┆ 0.962687 │
│ s__Prevotella sp000434975       ┆ 0.960199 │
│ s__Cryptobacteroides sp0004326… ┆ 0.956468 │
│ s__Ornithospirochaeta sp022785… ┆ 0.956157 │
│ s__Colivicinus sp002299675      ┆ 0.954913 │
│ s__CAG-170 sp002404795          ┆ 0.951182 │
│ s__Prevotella sp900548195       ┆ 0.95056  │
└─────────────────────────────────┴──────────┘


In [12]:
xx_df = (
    by_species
).filter(~(pl.col('species_name').is_in(CORE_NAMES)))
xx_df.filter(pl.col('freq') >= 0.95)

species_name,freq
str,f64
"""s__Prevotella sp900548195""",0.95056
"""s__Escherichia coli""",0.994403
"""s__CAG-170 sp002404795""",0.951182


In [13]:
xx_df = (
    by_species
).filter((pl.col('species_name').is_in(CORE_NAMES)))
xx_df.filter(pl.col('freq') >= 0.95)

species_name,freq
str,f64
"""s__Sodaliphilus sp004557565""",0.990361
"""s__Ornithospirochaeta sp022785…",0.956157
"""s__Cryptobacteroides sp0004349…",0.968595
"""s__Lactobacillus amylovorus""",0.976368
"""s__Fimisoma sp002320005""",0.973881
…,…
"""s__Cryptobacteroides sp0004326…",0.956468
"""s__Cryptobacteroides sp9005469…",0.978856
"""s__Cryptobacteroides sp0340892…",0.978856


In [14]:
all_foo = CORE_NAMES.union(SPECIES_ABOVE_95)

In [15]:
xx = []
for name in all_foo:
    is_core = 0
    is_gtdb95 = 0
    if name in CORE_NAMES:
        is_core = 1
    if name in SPECIES_ABOVE_95:
        is_gtdb95 = 1

    xx.append(dict(species_name=name, is_core=is_core, is_gtdb95=is_gtdb95))

xx2_df = pl.DataFrame(xx)

with pl.Config(tbl_rows=-1):
    print(xx2_df.join(by_species, on='species_name', how='inner').sort('freq'))

shape: (19, 4)
┌─────────────────────────────────┬─────────┬───────────┬──────────┐
│ species_name                    ┆ is_core ┆ is_gtdb95 ┆ freq     │
│ ---                             ┆ ---     ┆ ---       ┆ ---      │
│ str                             ┆ i64     ┆ i64       ┆ f64      │
╞═════════════════════════════════╪═════════╪═══════════╪══════════╡
│ s__UBA2868 sp004552595          ┆ 1       ┆ 0         ┆ 0.824316 │
│ s__JAFBIX01 sp021531895         ┆ 1       ┆ 0         ┆ 0.895522 │
│ s__Mogibacterium_A kristiansen… ┆ 1       ┆ 0         ┆ 0.918843 │
│ s__Floccifex porci              ┆ 1       ┆ 0         ┆ 0.924751 │
│ s__Bariatricus sp004560705      ┆ 1       ┆ 0         ┆ 0.939366 │
│ s__Prevotella sp900548195       ┆ 0       ┆ 1         ┆ 0.95056  │
│ s__CAG-170 sp002404795          ┆ 0       ┆ 1         ┆ 0.951182 │
│ s__Colivicinus sp002299675      ┆ 1       ┆ 1         ┆ 0.954913 │
│ s__Ornithospirochaeta sp022785… ┆ 1       ┆ 1         ┆ 0.956157 │
│ s__Cryptobacteroi

In [16]:
all_foo.join

AttributeError: 'set' object has no attribute 'join'